<a href="https://colab.research.google.com/github/KasumiMercury/yolo_person_finetuning_colab/blob/main/yolo_person_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/yolo26_person_ft"
RUNS_DIR   = f"{DRIVE_ROOT}/runs"          # 学習ログ/重み
WEIGHTS_DIR= f"{DRIVE_ROOT}/weights"       # 事前学習重みのキャッシュ
for p in [DRIVE_ROOT, RUNS_DIR, WEIGHTS_DIR]:
    os.makedirs(p, exist_ok=True)

print("DRIVE_ROOT =", DRIVE_ROOT)

In [ ]:
DATA_DIR = "/content/coco_person_yolo"   # ローカルSSDに展開

In [ ]:
import shutil
shutil.rmtree(DATA_DIR)

In [ ]:
import os, zipfile, shutil

ZIP_SRC  = f"{DRIVE_ROOT}/datasets/coco_person_yolo.zip"

if not os.path.isdir(DATA_DIR) or not os.listdir(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_SRC) as z:
        z.extractall(DATA_DIR)

import yaml
YAML_PATH = f"{DATA_DIR}/dataset.yaml"
with open(YAML_PATH, "w") as f:
    yaml.safe_dump({
        "path":  DATA_DIR,
        "train": "images/train",
        "val":   "images/val",
        "names": {0: "person"},
    }, f, sort_keys=False)
print(YAML_PATH)
print(open(YAML_PATH).read())

In [ ]:
!pip -q install -U ultralytics pycocotools onnx onnxruntime-gpu

import ultralytics
ultralytics.checks()

In [ ]:
import os, shutil
from ultralytics import YOLO

BASE_MODEL = "yolo26n.pt"   # or "yolo26s.pt"

cached = os.path.join(WEIGHTS_DIR, BASE_MODEL)
if not os.path.exists(cached):
    # 初回のみダウンロード→Driveへコピー
    m = YOLO(BASE_MODEL)    # DL
    src = m.ckpt_path if hasattr(m, "ckpt_path") else BASE_MODEL
    if os.path.exists(src):
        shutil.copy(src, cached)
print("pretrained:", cached)

In [ ]:
RUN_NAME   = "yolo26n_person_ft"
PROJECT    = RUNS_DIR

In [ ]:
# train reset

import shutil
shutil.rmtree(f"{PROJECT}/{RUN_NAME}", ignore_errors=True)

In [ ]:
# train
import os
from ultralytics import YOLO

LAST_CKPT  = f"{PROJECT}/{RUN_NAME}/weights/last.pt"

common_args = dict(
    data=YAML_PATH,
    epochs=300,
    imgsz=800,
    batch=32,            # Colab T4なら16〜32、A100なら64〜
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,       # 再実行で同じディレクトリに上書き継続
    patience=50,         # early stopping
    cache="disk",        # Driveにする場合は"disk"
    workers=8,
    device=0,
    # --- person特化/軽量化向けの調整 ---
    single_cls=True,     # 単一クラス学習(nc=1)
    cos_lr=True,         # コサインLR
    close_mosaic=20,     # 終盤はmosaic無効化で収束安定
    # augmentation: personは形状変化が大きいので軽めに調整
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    amp=True,
    lr0=0.015,
)

def is_resumable(ckpt_path):
    if not os.path.exists(ckpt_path):
        return False
    try:
        import torch
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        return ckpt.get("epoch", -1) >= 0 and ckpt.get("optimizer") is not None
    except Exception:
        return False

if is_resumable(LAST_CKPT):
    print(f"[RESUME] from {LAST_CKPT}")
    model = YOLO(LAST_CKPT)
    results = model.train(resume=True)        # 元args.yaml から復元
else:
    print(f"[NEW] from {cached} (last.pt not resumable or missing)")
    model = YOLO(cached)
    results = model.train(**common_args)      # common_args を渡す

In [ ]:
import os
from IPython.display import Image, display

RUN_DIR = f"{RUNS_DIR}/yolo26n_person_ft"
print(os.listdir(RUN_DIR))

for name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
             "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png", "labels.jpg"]:
    p = os.path.join(RUN_DIR, name)
    if os.path.exists(p):
        print(f"\n=== {name} ===")
        display(Image(p))

In [ ]:
# validate
from ultralytics import YOLO

BEST = f"{PROJECT}/{RUN_NAME}/weights/best.pt"

# ファインチューニング後
ft = YOLO(BEST)
metrics_ft = ft.val(data=YAML_PATH, imgsz=640, split="val")
print("FT  mAP50-95:", metrics_ft.box.map, " mAP50:", metrics_ft.box.map50)

# 元の事前学習モデル(80クラス)をperson-onlyのvalで比較
# classes=[0] でpersonだけ評価
base = YOLO(cached)
metrics_base = base.val(data=YAML_PATH, imgsz=640, split="val", classes=[0])
print("Base mAP50-95:", metrics_base.box.map, " mAP50:", metrics_base.box.map50)

In [ ]:
# export

from ultralytics import YOLO
model = YOLO(BEST)

# ONNX (WebやONNX Runtimeで使用)
onnx_path = model.export(format="onnx", imgsz=640, opset=13, dynamic=True, simplify=True)
print("ONNX:", onnx_path)

# TFLite INT8 (エッジデバイス向け・さらなる軽量化)
# tflite_path = model.export(format="tflite", int8=True, data=YAML_PATH, imgsz=640)
# print("TFLite:", tflite_path)

In [ ]:
# test

from ultralytics import YOLO
model = YOLO(BEST)
res = model.predict(
    source=f"{DATA_DIR}/images/val",
    imgsz=640, conf=0.25,
    save=True,
    project=PROJECT, name=f"{RUN_NAME}_pred", exist_ok=True,
)
print("saved to:", res[0].save_dir)